# Colab: tuning and training (Stages 3-4)**Run Stages 1-2 on your own machine first.** Ingest needs the full raw dataset(tens of GB) and is disk-bound, not GPU-bound; uploading it here would wastehours and will not fit in a free 15 GB Drive.What you upload instead is the single processed file `internet_matrix.npy`(~341 MB) plus three tiny metadata files. That is all the models need.Before starting: **Runtime -> Change runtime type -> T4 GPU**.

In [ ]:
# 1. Confirm a GPU is actually attached. If this errors, fix the runtime type.!nvidia-smi

## 2. Mount DrivePut the four processed files in `MyDrive/milan/processed/` before running this:```internet_matrix.npy      <- the only large one (~341 MB)timestamps_ns.npysquare_ids.npymeta.json```They are produced on your laptop by `make data`, in `data/processed/`.You do **not** need `observed_mask.npy` for training.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')DRIVE = '/content/drive/MyDrive/milan'!mkdir -p {DRIVE}/processed {DRIVE}/results!ls -lh {DRIVE}/processed

## 3. Clone the repository and install dependencies

In [ ]:
REPO = 'https://github.com/<your-username>/milan-traffic-forecasting.git'%cd /content![ -d milan-traffic-forecasting ] || git clone $REPO%cd /content/milan-traffic-forecasting!git pull --quiet || true# torch is preinstalled on Colab; statsmodels usually needs an upgrade.!pip install -q --upgrade statsmodels pyyaml psutilimport torchprint('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

In [ ]:
# Sanity check before spending GPU time.!python -m pytest tests/ -q

## 4. Colab configData is read from Drive and **results are written back to Drive**, so adisconnect loses nothing. Absolute paths in the config override the projectroot, which is what makes this work.Edit `sequence_length` and the `tuning` grids here rather than in`configs/config.yaml`, so your laptop config stays untouched.

In [ ]:
config_text = f'''seed: 42paths:  raw_dir: data/raw  processed_dir: {DRIVE}/processed  results_dir: {DRIVE}/results  figures_dir: {DRIVE}/results/figures  tables_dir: {DRIVE}/results/tables  predictions_dir: {DRIVE}/results/predictionsdataset:  activity: internet  n_squares: 10000  intervals_per_day: 144  timezone: Europe/Rome  chunksize: 2000000  missing_policy: interpolatesplits:  train_start: "2013-11-01 00:00"  train_end:   "2013-12-08 23:50"  val_start:   "2013-12-09 00:00"  val_end:     "2013-12-15 23:50"  test_start:  "2013-12-16 00:00"  test_end:    "2013-12-22 23:50"eda:  extra_squares: [4159, 4556]  first_two_weeks_start: "2013-11-01 00:00"  first_two_weeks_end:   "2013-11-14 23:50"  acf_max_lag: 1100forecasting:  horizon: 1  sequence_length: 144  use_time_features: true  scaler:    log1p: true    method: standardmodels:  baselines:    seasonal_period: 144  sarimax:    order: [2, 0, 1]    fourier:      - {{period: 144,  n_terms: 5}}      - {{period: 1008, n_terms: 3}}    log1p: true    trend: c    maxiter: 200  lstm:    hidden_size: 64    num_layers: 2    dropout: 0.2    lr: 0.001    batch_size: 128    max_epochs: 60    patience: 8    grad_clip: 1.0    loss: huber  tcn:    channels: [32, 32, 32, 32]    kernel_size: 3    dropout: 0.15    lr: 0.002    batch_size: 128    max_epochs: 60    patience: 8    grad_clip: 1.0    loss: hubertuning:  lstm:    sequence_length: [24, 144, 288]    hidden_size: [32, 64, 128]    num_layers: [1, 2]    lr: [0.003, 0.001]  tcn:    sequence_length: [144, 288]    kernel_size: [3, 5]    dropout: [0.1, 0.2]    lr: [0.003, 0.001]  sarimax:    order: [[1, 0, 0], [2, 0, 1], [3, 0, 2], [1, 1, 1]]timing:  n_repeats: 3  warmup: 1'''open('configs/config_colab.yaml', 'w').write(config_text)print('written')

## 5. Which square are we tuning on?Stage 2 on your laptop cached the top three to`results/tables/top3.json`. Copy that file into `{DRIVE}/results/tables/`, orjust set the IDs by hand below.

In [ ]:
import json, ostop3_path = f'{DRIVE}/results/tables/top3.json'if os.path.exists(top3_path):    print('top 3 areas:', json.load(open(top3_path))['top3'])else:    print('top3.json not found -- upload it, or pass --squares explicitly below.')

## 6. Grid search (the reason you are on a GPU)**Safe to re-run after a disconnect.** Every trial is flushed to`experiment_log.csv` on Drive as it completes, and completed trials are skippedon the next run. If the session drops at trial 20 of 36, just run this cellagain.The full LSTM grid is 36 configurations. Start with `--max-trials 8` to measurethe per-trial cost before committing.

In [ ]:
!python scripts/03_run_experiments.py --config configs/config_colab.yaml --tune --max-trials 8

In [ ]:
# Once you know the pace, run the rest. Re-run freely; it resumes.!python scripts/03_run_experiments.py --config configs/config_colab.yaml --tune

In [ ]:
import pandas as pdlog = pd.read_csv(f'{DRIVE}/results/tables/experiment_log.csv')display(log.sort_values('val_MAE').head(15))print('\nFill in the `rationale` column by hand before submitting -- '      'the brief requires documented reasoning for each adjustment.')

## 7. Final runsCopy the winning hyperparameters into the `models:` block of`configs/config_colab.yaml` above, re-run that cell, then run this.

In [ ]:
!python scripts/03_run_experiments.py --config configs/config_colab.yaml

In [ ]:
!python scripts/04_failure_analysis.py --config configs/config_colab.yaml

## 8. Collect the resultsEverything is already on Drive, but a zip is easier to pull down in one go.

In [ ]:
!cd {DRIVE} && zip -rq results_colab.zip results && ls -lh {DRIVE}/results_colab.zipfrom google.colab import filesfiles.download(f'{DRIVE}/results_colab.zip')

## Notes- **Keep the tab open.** Free Colab disconnects after ~90 minutes idle and caps  sessions at ~12 hours. Resume support covers you, but it cannot resume a  single trial that was mid-flight.- **SARIMAX gets no GPU benefit.** `statsmodels` is CPU-only; run it on your  laptop if you would rather save GPU quota.- **If you hit the GPU limit**, the same commands run on a CPU runtime, just  slower. The TCN is far less affected than the LSTM, because convolutions  parallelise over the time axis while recurrence does not.- **Check the timings you report.** Task 4-IV asks for the hardware the numbers  were recorded on. If you tune on a T4 and report those times, say T4 — and if  you mix machines across models, the comparison stops being meaningful.